In [ ]:
import netgen.geom2d as geom2d
from ngsolve import *
import netgen.gui
%gui tk

In [ ]:
#Geometry Generation

geo = geom2d.SplineGeometry()
geo.AddRectangle((0, 0), (2.2, 0.41), leftdomain=1, rightdomain=0, bcs=["bottom", "right", "top", "left"])

geo.AddCircle(c=(0.2, 0.2), r=0.05, leftdomain=0, rightdomain=1, bc="cylinder", maxh=0.01)

mesh = Mesh(geo.GenerateMesh(maxh=0.025))
mesh.Curve(3)

Draw(mesh)

In [ ]:
# Physical parameters
nu = 0.00007   # Kinematic viscosity
tau = 0.0075   # Time step
t = 0.0       # Current time
tend = 10    # End time (start shorter for debugging)

# Finite element spaces (Taylor-Hood: P2 velocity, P1 pressure)
V = VectorH1(mesh, order=3, dirichlet="left|bottom|top|cylinder")
Q = H1(mesh, order=2, dirichlet="right")
X = V * Q

# Solution
gfu = GridFunction(X)
u, p = gfu.components

# Boundary condition: parabolic inflow at left
uin = CoefficientFunction((1.5 * 4.0 * y * (0.41 - y) / (0.41**2), 0))

# Previous time step
gfuold = GridFunction(X)
uold = gfuold.components[0]

# Weak formulation
(du, dp), (v, q) = X.TnT()

a = BilinearForm(X)
a += du * v * dx                                       # Time derivative
a += tau * InnerProduct(grad(du) * uold, v) * dx       # Convection (frozen uold)
a += tau * nu * InnerProduct(grad(du), grad(v)) * dx   # Viscosity
a += -tau * dp * div(v) * dx                           # Pressure
a += -tau * div(du) * q * dx                           # Incompressibility

f = LinearForm(X)
f += uold * v * dx

# Visualization
from ngsolve import Draw
vel_mag = sqrt(u[0]**2 + u[1]**2)
Draw(vel_mag, mesh, "velocity", min=0, max=2, autoscale=False)

# initial state
gfu.vec[:] = 0
u.Set(uin, definedon=mesh.Boundaries("left"))
gfuold.vec.data = gfu.vec

# snapshot storage for animation (velocity field over time)
save_every = 10
vel_history = GridFunction(V, multidim=0)
vel_history.AddMultiDimComponent(u.vec)  # frame at t=0

step = 0

# Time loop
while t < tend:
    gfuold.vec.data = gfu.vec

    a.Assemble()
    f.Assemble()

    u.Set(uin, definedon=mesh.Boundaries("left"))
    inv = a.mat.Inverse(X.FreeDofs(), inverse="umfpack")
    res = f.vec.CreateVector()
    res.data = f.vec - a.mat * gfu.vec
    gfu.vec.data += inv * res

    t += tau
    step += 1

    if step % 1 == 0:
        vel_history.AddMultiDimComponent(u.vec)

    if step % 1 == 0:
        inlet_v = u(mesh(0.01, 0.205))
        outlet_v = u(mesh(2.19, 0.205))
        u_l2 = sqrt(Integrate(InnerProduct(u, u), mesh))
        div_l2 = sqrt(Integrate(div(u) * div(u), mesh))

        print(
            f"t={t:.2f} | Inlet:{inlet_v[0]:.3f} | Outlet:{outlet_v[0]:.3f} "
            f"| ||u||_L2:{u_l2:.3f} | ||div u||_L2:{div_l2:.3e}"
        )
        Redraw()

print(f"Final time: {t}")

In [ ]:
# Animation from vel_history

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
import matplotlib.animation as animation
from IPython.display import Image, display

 
pnts = np.array([v.point for v in mesh.vertices])
tris  = np.array([[v.nr for v in el.vertices] for el in mesh.Elements(VOL)])
triang = mtri.Triangulation(pnts[:, 0], pnts[:, 1], tris)


n_frames = len(vel_history.vecs)
print(f"Total frames stored: {n_frames}")

def get_velocity_magnitude(frame_idx):
    """Reconstruct velocity magnitude at mesh vertices for a given frame."""
    tmp = GridFunction(V)
    tmp.vec.data = vel_history.vecs[frame_idx]
    vals = np.array([
        np.sqrt(tmp(mesh(*v.point))[0]**2 + tmp(mesh(*v.point))[1]**2)
        for v in mesh.vertices
    ])
    return vals

print("Pre-computing frames (this may take a while)...")
all_vals = []
for i in range(n_frames):
    all_vals.append(get_velocity_magnitude(i))
    if (i + 1) % 20 == 0 or i == n_frames - 1:
        print(f"  Frame {i+1}/{n_frames}")

vmin = 0
vmax = max(np.max(v) for v in all_vals)
print(f"Velocity range: [{vmin}, {vmax:.3f}]")


fig, ax = plt.subplots(figsize=(14, 3), dpi=120)
fig.subplots_adjust(bottom=0.18, top=0.88)

tc = ax.tripcolor(triang, all_vals[0], shading='gouraud',
                  cmap='jet', vmin=vmin, vmax=vmax)
fig.colorbar(tc, ax=ax, label='|u| [m/s]', shrink=0.9)
ax.set_aspect('equal')
ax.set_xlabel('x')
ax.set_ylabel('y')

def animate(i):
    ax.clear()
    ax.tripcolor(triang, all_vals[i], shading='gouraud',
                 cmap='jet', vmin=vmin, vmax=vmax)
    ax.set_aspect('equal')
    frame_time = i * save_every * tau
    ax.set_title(f'Velocity magnitude  —  t = {frame_time:.2f} s')
    ax.set_xlabel('x')
    ax.set_ylabel('y')

save_every = 1
ani = animation.FuncAnimation(fig, animate, frames=n_frames,
                              interval=1, blit=False)


In [ ]:
import os
os.chdir('/Users/saryabou/Documents/GitHub/Scientific_Computing/Scientific-computing/set_3')

In [ ]:
# Save as GIF 
gif_path = 'cfd_animation.gif'
print(f"Saving {n_frames}-frame animation to {gif_path} ...")
ani.save(gif_path, writer='pillow', fps= 300)
plt.close(fig)
print("Done!")

display(Image(filename=gif_path))

In [ ]:
from PIL import Image as PILImage
import os
gif_path = 'cfd_animation.gif'
os.makedirs("frames", exist_ok=True)
gif = PILImage.open(gif_path) 
total = gif.n_frames
n_extract = 6
indices = [int(i * (total - 1) / (n_extract - 1)) for i in range(n_extract)]

for idx in indices:
    gif.seek(idx)
    frame = gif.convert("RGB")
    frame.save(f"frames/fem_frame_{idx:04d}.png", dpi=(150, 150))
    print(f"  Saved frames/fem_frame_{idx:04d}.png")

print(f"\nExtracted {n_extract} frames to frames/")